In [1]:
from pyspark.sql import SparkSession
from collections import Counter
from pyspark.sql.functions import sum, desc, countDistinct, year, month, avg, datediff, round, dense_rank
from pyspark.sql.window import Window
import os
import builtins



In [2]:
spark = (
    SparkSession.builder
    .appName("TradeCorpTransformations")
    .getOrCreate()
)

PATH = "/home/jovyan/data/tmp"

In [3]:
df_categories = spark.read.parquet(f"{PATH}/categories.parquet")
df_customers = spark.read.parquet(f"{PATH}/customers.parquet")
df_employees = spark.read.parquet(f"{PATH}/employees.parquet")
df_order_details = spark.read.parquet(f"{PATH}/order_details.parquet")
df_orders = spark.read.parquet(f"{PATH}/orders.parquet")
df_products = spark.read.parquet(f"{PATH}/products.parquet")
df_shippers = spark.read.parquet(f"{PATH}/shippers.parquet")
df_suppliers = spark.read.parquet(f"{PATH}/suppliers.parquet")

## Q21.

In [4]:
df_orders_customers = (
    df_orders
    .join(df_customers, on="customer_id", how="inner")
    .select(
        "order_id",
        "company_name",
        "country",
        "order_date",
        "freight"
    )
)

## Q22.

In [5]:
df_order_details_products = (
    df_order_details
    .join(
        df_products.select(
            "product_id",
            "product_name",
            "category_id",
            "unit_price"
        ),
        on="product_id",
        how="inner"
    )
)

## Q23.

In [6]:
df_products_categories = (
    df_products
    .join(
        df_categories.select(
            "category_id",
            "category_name",
            "description"
        ),
        on="category_id",
        how="inner"
    )
)

## Q24A.

In [7]:
df_orders_enrichi_a = (
    df_order_details
    .join(df_orders, on="order_id", how="inner")
    .join(df_customers, on="customer_id", how="inner")
    .join(df_products_categories, on="product_id", how="inner")
    .join(df_employees, on="employee_id", how="inner")
    .join(df_shippers, on="shipper_id", how="inner")
)

In [8]:
column_counts = Counter(df_orders_enrichi_a.columns)

duplicates = [
    column
    for column, count in column_counts.items()
    if count > 1
]

print("Colonnes en double :", duplicates)

Colonnes en double : ['company_name', 'city', 'country', 'phone']


## Q24B.

In [9]:
df_customers_renomme = (
    df_customers
    .withColumnRenamed("company_name", "customer_company_name")
    .withColumnRenamed("city", "customer_city")
    .withColumnRenamed("country", "customer_country")
    .withColumnRenamed("phone", "customer_phone")
)

df_employees_renomme = (
    df_employees
    .withColumnRenamed("city", "employee_city")
    .withColumnRenamed("country", "employee_country")
)

df_shippers_renomme = (
    df_shippers
    .withColumnRenamed("company_name", "shipper_name")
    .withColumnRenamed("phone", "shipper_phone")
)

In [10]:
df_orders_enriched = (
    df_order_details
    .join(df_orders, on="order_id", how="inner")
    .join(df_customers_renomme, on="customer_id", how="inner")
    .join(df_products_categories, on="product_id", how="inner")
    .join(df_employees_renomme, on="employee_id", how="inner")
    .join(df_shippers_renomme, on="shipper_id", how="inner")
)

In [11]:
column_counts = Counter(df_orders_enriched.columns)

duplicates = [
    column
    for column, count in column_counts.items()
    if count > 1
]

print("Colonnes encore en double :", duplicates)

Colonnes encore en double : []


## Q25.

In [12]:
df_ca_clients = (
    df_orders_enriched
    .groupBy("customer_company_name")
    .agg(
        sum("sous_total").alias("ca_total")
    )
    .orderBy(desc("ca_total"))
)

df_ca_clients.show(10, truncate=False)

+----------------------------+------------------+
|customer_company_name       |ca_total          |
+----------------------------+------------------+
|QUICK-Stop                  |51682.73999999999 |
|Save-a-lot Markets          |40238.09          |
|Ernst Handel                |39975.91          |
|Mère Paillarde              |22871.070000000003|
|Rattlesnake Canyon Grocery  |17636.1           |
|Simons bistro               |16232.42          |
|Hungry Owl All-Night Grocers|14403.03          |
|Folk och fä HB              |13200.92          |
|HILARION-Abastos            |11799.74          |
|Berglunds snabbköp          |11758.92          |
+----------------------------+------------------+
only showing top 10 rows


## Q26.

In [13]:
df_ca_categories = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        sum("sous_total").alias("ca_total"),
        countDistinct("product_id").alias("nb_produits_distincts")
    )
    .orderBy(desc("ca_total"))
)

df_ca_categories.show(truncate=False)

+--------------+------------------+---------------------+
|category_name |ca_total          |nb_produits_distincts|
+--------------+------------------+---------------------+
|Dairy Products|108086.90000000002|9                    |
|Beverages     |90368.64          |9                    |
|Confections   |82657.78000000001 |13                   |
|Seafood       |66959.23          |12                   |
|Condiments    |54995.0           |11                   |
|Grains/Cereals|51463.630000000005|6                    |
|Produce       |40992.09          |4                    |
|Meat/Poultry  |11017.17          |2                    |
+--------------+------------------+---------------------+



## Q27.

In [14]:
df_ca_mensuel = (
    df_orders_enriched
    .groupBy(
        year("order_date").alias("annee"),
        month("order_date").alias("mois")
    )
    .agg(
        sum("sous_total").alias("ca_total")
    )
    .orderBy("annee", "mois")
)

df_ca_mensuel.show(truncate=False)

+-----+----+------------------+
|annee|mois|ca_total          |
+-----+----+------------------+
|1997 |1   |51487.509999999995|
|1997 |2   |31549.039999999997|
|1997 |3   |33226.33          |
|1997 |4   |41510.59999999999 |
|1997 |5   |48895.270000000004|
|1997 |6   |29875.469999999998|
|1997 |7   |45162.87999999999 |
|1997 |8   |38039.92999999999 |
|1997 |9   |43335.42999999999 |
|1997 |10  |48574.5           |
|1997 |11  |39898.78          |
|1997 |12  |54984.700000000004|
+-----+----+------------------+



## Q28.

In [15]:
df_performance_employes = (
    df_orders_enriched
    .groupBy("full_name")
    .agg(
        countDistinct("order_id").alias("nb_commandes"),
        round(sum("sous_total"), 2).alias("ca_total"),
        round(
            avg(datediff("shipped_date", "order_date")),
            2
        ).alias("delai_moyen_livraison_jours")
    )
)

df_performance_employes.show(truncate=False)

+----------------+------------+---------+---------------------------+
|full_name       |nb_commandes|ca_total |delai_moyen_livraison_jours|
+----------------+------------+---------+---------------------------+
|Anne Dodsworth  |18          |20595.99 |10.03                      |
|Nancy Davolio   |54          |81898.92 |7.84                       |
|Andrew Fuller   |40          |54907.03 |10.19                      |
|Steven Buchanan |18          |17185.65 |6.46                       |
|Janet Leverling |71          |97081.27 |8.92                       |
|Robert King     |33          |49562.78 |9.81                       |
|Laura Callahan  |53          |47077.95 |7.95                       |
|Margaret Peacock|75          |104193.78|8.3                        |
|Michael Suyama  |33          |34037.07 |7.94                       |
+----------------+------------+---------+---------------------------+



## Q29.

In [16]:
# Calcul du CA par produit et par catégorie
df_ca_produits = (
    df_orders_enriched
    .groupBy(
        "category_name",
        "product_id",
        "product_name"
    )
    .agg(
        sum("sous_total").alias("ca")
    )
)

# Fenêtre partitionnée par catégorie
window_category = (
    Window
    .partitionBy("category_name")
    .orderBy(desc("ca"))
)

# Classement des produits dans chaque catégorie
df_ranking_produits = (
    df_ca_produits
    .withColumn(
        "rang",
        dense_rank().over(window_category)
    )
    .orderBy("category_name", "rang")
)

df_ranking_produits.show(truncate=False)

+-------------+----------+--------------------------------+-----------------+----+
|category_name|product_id|product_name                    |ca               |rang|
+-------------+----------+--------------------------------+-----------------+----+
|Beverages    |38        |Côte de Blaye                   |49198.09         |1   |
|Beverages    |43        |Ipoh Coffee                     |11069.9          |2   |
|Beverages    |76        |Lakkalikööri                    |7379.1           |3   |
|Beverages    |70        |Outback Lager                   |5468.4           |4   |
|Beverages    |35        |Steeleye Stout                  |5274.9           |5   |
|Beverages    |75        |Rhönbräu Klosterbier            |4485.55          |6   |
|Beverages    |39        |Chartreuse verte                |4475.700000000001|7   |
|Beverages    |34        |Sasquatch Ale                   |2107.0           |8   |
|Beverages    |67        |Laughing Lumberjack Lager       |910.0            |9   |
|Con

## Q30.

In [17]:
# CA par mois
df_ca_mensuel = (
    df_orders_enriched
    .groupBy(
        year("order_date").alias("annee"),
        month("order_date").alias("mois")
    )
    .agg(
        sum("sous_total").alias("ca_mensuel")
    )
)

# Fenêtre 
window_cumul = (
    Window
    .orderBy("annee", "mois")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# CA cumulé
df_ca_cumule = (
    df_ca_mensuel
    .withColumn(
        "ca_cumule",
        sum("ca_mensuel").over(window_cumul)
    )
    .orderBy("annee", "mois")
)

df_ca_cumule.show(truncate=False)

+-----+----+------------------+------------------+
|annee|mois|ca_mensuel        |ca_cumule         |
+-----+----+------------------+------------------+
|1997 |1   |51487.509999999995|51487.509999999995|
|1997 |2   |31549.039999999997|83036.54999999999 |
|1997 |3   |33226.33          |116262.87999999999|
|1997 |4   |41510.59999999999 |157773.47999999998|
|1997 |5   |48895.270000000004|206668.75         |
|1997 |6   |29875.469999999998|236544.22         |
|1997 |7   |45162.87999999999 |281707.1          |
|1997 |8   |38039.92999999999 |319747.02999999997|
|1997 |9   |43335.42999999999 |363082.45999999996|
|1997 |10  |48574.5           |411656.95999999996|
|1997 |11  |39898.78          |451555.74         |
|1997 |12  |54984.700000000004|506540.44         |
+-----+----+------------------+------------------+



## Q31.

In [18]:
# 5 produits les plus vendus en quantité
df_top_produits = (
    df_orders_enriched
    .groupBy("product_id", "product_name")
    .agg(
        sum("quantite").alias("quantite_totale")
    )
    .orderBy(desc("quantite_totale"))
    .limit(5)
)

df_top_produits.show(truncate=False)

+----------+----------------------+---------------+
|product_id|product_name          |quantite_totale|
+----------+----------------------+---------------+
|56        |Gnocchi di nonna Alice|971            |
|59        |Raclette Courdavault  |752            |
|60        |Camembert Pierrot     |665            |
|75        |Rhönbräu Klosterbier  |630            |
|21        |Sir Rodney's Scones   |610            |
+----------+----------------------+---------------+



In [19]:
# 3 pays clients générant le plus de chiffre d'affaires
df_top_pays = (
    df_orders_enriched
    .groupBy("customer_country")
    .agg(
        sum("sous_total").alias("ca_total")
    )
    .orderBy(desc("ca_total"))
    .limit(3)
)

df_top_pays.show(truncate=False)

+----------------+------------------+
|customer_country|ca_total          |
+----------------+------------------+
|GERMANY         |100641.29000000002|
|USA             |90731.70999999998 |
|AUSTRIA         |46559.49          |
+----------------+------------------+



## Q32.

In [20]:
df_orders_enriched.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/output/orders_enriched.parquet")

## Q33.

In [21]:
# Relire le fichier Parquet
df_orders_enriched_parquet = spark.read.parquet(
    "/home/jovyan/data/output/orders_enriched.parquet"
)

# Comparer le nombre de lignes
nb_original = df_orders_enriched.count()
nb_parquet = df_orders_enriched_parquet.count()

print("Nombre de lignes original :", nb_original)
print("Nombre de lignes Parquet :", nb_parquet)

if nb_original == nb_parquet:
    print("Le nombre de lignes est identique.")
else:
    print("Le nombre de lignes est différent.")

df_orders_enriched_parquet.printSchema()

Nombre de lignes original : 893
Nombre de lignes Parquet : 893
Le nombre de lignes est identique.
root
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customer_company_name: string (nullable = 

## Q34.

In [22]:
csv_path = "/home/jovyan/data/raw/orders.csv"
parquet_path = "/home/jovyan/data/output/orders_enriched.parquet"

# Taille du CSV
csv_size = os.path.getsize(csv_path)

# Taille totale du dossier Parquet
parquet_size = builtins.sum(
    os.path.getsize(os.path.join(root, file))
    for root, _, files in os.walk(parquet_path)
    for file in files
)

print(f"Taille CSV     : {csv_size / 1024:.2f} Ko")
print(f"Taille Parquet : {parquet_size / 1024:.2f} Ko")

Taille CSV     : 98.76 Ko
Taille Parquet : 71.58 Ko


Le format Parquet est plus efficace que CSV car :

- il utilise un stockage en colonnes
- il applique de la compression 
- il conserve les types de données 
Spark peut lire uniquement les colonnes nécessaires au lieu de parcourir toutes les données 

## Q35.

In [23]:
output_path = "/home/jovyan/data/output/orders_partitioned"

df_orders_enriched.write \
    .mode("overwrite") \
    .partitionBy("customer_country") \
    .parquet(output_path)

In [24]:
# OBSERVATION DES FICHIERS
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

orders_partitioned/
    customer_country=ARGENTINA/
    customer_country=AUSTRIA/
    customer_country=BELGIUM/
    customer_country=BRAZIL/
    customer_country=CANADA/
    customer_country=DENMARK/
    customer_country=FINLAND/
    customer_country=FRANCE/
    customer_country=GERMANY/
    customer_country=IRELAND/
    customer_country=ITALY/
    customer_country=MEXICO/
    customer_country=NORWAY/
    customer_country=POLAND/
    customer_country=PORTUGAL/
    customer_country=SPAIN/
    customer_country=SWEDEN/
    customer_country=SWITZERLAND/
    customer_country=UK/
    customer_country=USA/
    customer_country=VENEZUELA/


## Q36.

In [25]:
jdbc_url = "jdbc:postgresql://postgres:5432/tradecorp"

properties = {
    "user": "tradecorp",
    "password": "tradecorp",
    "driver": "org.postgresql.Driver"
}

df_orders_enriched.write \
    .mode("overwrite") \
    .jdbc(
        url=jdbc_url,
        table="orders_enriched",
        properties=properties
    )